# Validación de datos
Compara una tabla generada en PySpark contra su equivalente de Dataflow Gen2.

La validación incluye:
- Esquema (columnas y tipos)
- Volumen (cantidad de filas)
- Contenido (diferencias entre filas)

Actualiza los nombres de tabla en la última línea antes de ejecutar.

In [ ]:

def comparar_notebook_vs_dataflow(tabla_notebook, tabla_dataflow):
    print("Iniciando comparacion...")
    print(f"A: {tabla_notebook} (PySpark)")
    print(f"B: {tabla_dataflow} (Dataflow Gen2)\n")

    try:
        df_nb = spark.read.table(tabla_notebook)
        df_df = spark.read.table(tabla_dataflow)
    except Exception as e:
        print(f"Error al leer las tablas: {e}")
        return

    # 1. Comparacion de esquema (nombres de columnas y tipos)
    schema_nb = set(df_nb.schema.simpleString().split(","))
    schema_df = set(df_df.schema.simpleString().split(","))

    if schema_nb == schema_df:
        print("Esquemas identicos: ambas tablas tienen las mismas columnas y tipos.")
    else:
        diff = schema_nb.symmetric_difference(schema_df)
        print(f"Discrepancia en el esquema. Diferencias detectadas: {diff}")

    # 2. Comparacion de volumen
    count_nb = df_nb.count()
    count_df = df_df.count()

    if count_nb == count_df:
        print(f"Conteo de filas coincidente: {count_nb} registros.")
    else:
        print(f"Diferencia en filas: Notebook tiene {count_nb} y Dataflow {count_df}.")

    # 3. Comparacion de contenido (data diff)
    # Restamos los DataFrames para ver si hay filas que existen en uno pero no en otro.
    diff_nb_df = df_nb.subtract(df_df)
    diff_df_nb = df_df.subtract(df_nb)

    count_diff_nb_df = diff_nb_df.count()
    count_diff_df_nb = diff_df_nb.count()

    if count_diff_nb_df == 0 and count_diff_df_nb == 0:
        print("Contenido exacto: los datos son 100% identicos en ambas tablas.")
    else:
        print("El contenido no es identico.")
        if count_diff_nb_df > 0:
            print(f"- Filas en Notebook que no estan en Dataflow: {count_diff_nb_df}")
            diff_nb_df.show(5)
        if count_diff_df_nb > 0:
            print(f"- Filas en Dataflow que no estan en Notebook: {count_diff_df_nb}")
            diff_df_nb.show(5)

# Ejecucion de la comparativa
# Asegurate de usar los nombres de tabla que definiste en cada proceso
comparar_notebook_vs_dataflow("dbo.pyspark_sensor_data", "dbo.dataflow_sensor_data")